## **집계연산**

### **PySpark 설정**

In [1]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext

In [2]:
# [+] SparkConf, SparkContext 객체 생성하기
conf = SparkConf().setMaster('local').setAppName("reducation-operations")

In [3]:
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 04:04:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **reduce( ) 메서드**

In [7]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x, y : x + y)

15

In [8]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()

[1, 2, 3, 4, 5]

In [11]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [10]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
rdd.reduce(lambda x , y : x + y)

15

In [9]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5]).reduce(lambda x, y : (x * 2) + y)

57

In [6]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

In [10]:
sc.parallelize([1,2,3,4,5], 3).reduce(lambda x,y: (x * 2) + y)

31

In [11]:
sc.parallelize([1,2,3,4,5], 4).reduce(lambda x,y: (x * 2) + y)

35

### **10000 개 정수 난수의 총합 계산하기**

In [4]:
!pip install numpy

In [14]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [16]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0, 100, size=10000)
x[:100]

array([90, 27, 35, 64, 94, 17, 57, 10, 77, 20, 91, 86, 80, 65, 59, 74, 77,
       88, 36, 94, 18, 28, 95, 36,  1, 45, 28, 26, 89, 47, 50, 82, 52,  1,
       93, 53, 61, 51, 37, 39, 74, 57, 52, 90, 32, 88, 91, 47, 89,  9, 31,
        8, 26, 48, 52, 43, 42, 28, 75, 76,  5, 46, 23,  5, 81, 27, 64, 33,
        0, 39, 94, 56, 91, 27, 75, 44, 74, 25, 99, 55, 25, 17,  2, 97, 14,
       59, 37, 80, 81, 94, 42, 12, 12, 89, 49, 38, 79, 75, 62,  5])

In [17]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x, 8)

In [18]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

Partition 0: [np.int64(90), np.int64(27), np.int64(35), np.int64(64), np.int64(94), np.int64(17), np.int64(57), np.int64(10), np.int64(77), np.int64(20), np.int64(91), np.int64(86), np.int64(80), np.int64(65), np.int64(59), np.int64(74), np.int64(77), np.int64(88), np.int64(36), np.int64(94), np.int64(18), np.int64(28), np.int64(95), np.int64(36), np.int64(1), np.int64(45), np.int64(28), np.int64(26), np.int64(89), np.int64(47), np.int64(50), np.int64(82), np.int64(52), np.int64(1), np.int64(93), np.int64(53), np.int64(61), np.int64(51), np.int64(37), np.int64(39), np.int64(74), np.int64(57), np.int64(52), np.int64(90), np.int64(32), np.int64(88), np.int64(91), np.int64(47), np.int64(89), np.int64(9), np.int64(31), np.int64(8), np.int64(26), np.int64(48), np.int64(52), np.int64(43), np.int64(42), np.int64(28), np.int64(75), np.int64(76), np.int64(5), np.int64(46), np.int64(23), np.int64(5), np.int64(81), np.int64(27), np.int64(64), np.int64(33), np.int64(0), np.int64(39), np.int64(94),

In [19]:
# [+] x+y 집계 연산
rdd.reduce(lambda x, y : x + y)

np.int64(497485)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [20]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x, y : x + y)

12

In [21]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4] , 2).fold(1, lambda x, y : x + y)

13

In [22]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x , y : x * y)

0

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [ ]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([3,4, 10, 30, 12, 213]).groupBy(lambda x : x % 2).collect()

In [6]:
# [+] groupBy() 결과 출력하기
for k, v in res:
    print(k, list(v))

1 [3, 213]
0 [4, 10, 30, 12]


### **aggregate( ) 메서드**

In [ ]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4])\
    .reduce(lambda x, y: (x + y, x * y)) 
    # single value -> pair 생성 시도할 경우 오류

26/10/08 04:19:51 ERROR Executor: Exception in task 0.0 in stage 6.0 (TID 7)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/rdd.py", line 1922, in func
    yield reduce(f, iterator, initial)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/util.py", line 83, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2107/674806899.py", line 3, in <lambda>
TypeError: can only concatenate tuple (not "int") to tuple

	

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 6.0 failed 1 times, most recent failure: Lost task 0.0 in stage 6.0 (TID 7) (bbec1b65c823 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/rdd.py", line 1922, in func
    yield reduce(f, iterator, initial)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/py

In [8]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

TypeError: unsupported operand type(s) for +: 'int' and 'tuple'

In [9]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [13]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)

(10, 4)